In [1]:
import re
import datetime as dt
import pandas as pd
import pyreadr
import numpy as np
from datetime import timedelta, datetime
import warnings
warnings.filterwarnings("ignore")
import gc
gc.collect()
gc.collect()
gc.collect()

0

In [2]:
df= pd.read_csv(r"D:\TASK\HL & LAP\LAP\LAP files\LAP MOM\LAP_All_disb_agreements_2006_Nov'2025_Final.csv")
df= df[['AGREEMENTNO','EFF_RATE','LTV_bin', "AMTFIN","AGR AUTH DATE","STATUS_DESC","BRANCHDESC",
        "Region","State","Ticket size_bin", "Zone", "Area", "BCG INDUSTRYDESC",
        "BCG SUB INDUSTRYDESC","PROMOTIONDESC"]]
print(df.shape)
df.head(3)

(194886, 15)


,AGREEMENTNO,EFF_RATE,LTV_bin,AMTFIN,AGR AUTH DATE,STATUS_DESC,BRANCHDESC,Region,State,Ticket size_bin,Zone,Area,BCG INDUSTRYDESC,BCG SUB INDUSTRYDESC,PROMOTIONDESC
0,X0HECHE00000016515,12.00,4. 50%-60%,2400000,26/10/2006,CLOSED,CHENNAI ONE HE,TAMIL NADU,TAMIL NADU,2. 20L-50L,SOUTH 1,CHENNAI 1 AREA,Consultancy / Professional Services,Photo studio and AV,NaN
1,X0HECHE00000016528,14.85,4. 50%-60%,2400000,26/10/2006,CLOSED,CHENNAI ONE HE,TAMIL NADU,TAMIL NADU,2. 20L-50L,SOUTH 1,CHENNAI 1 AREA,Consultancy / Professional Services,Photo studio and AV,NaN
2,X0HECHE00000021140,15.15,2. 20%-40%,600000,14/11/2006,CLOSED,CHENNAI ONE HE,TAMIL NADU,TAMIL NADU,1. <=20L,SOUTH 1,CHENNAI 1 AREA,Consultancy / Professional Services,Photo studio and AV,NaN


In [3]:
# import pandas as pd
# df2= pd.read_excel(r"D:\TASK\LAP MOM Dashboard\Dashboard_data\LAP_MOM_Disbst.xlsx")
# df2= df2[["AGREEMENTNO","AMTFIN","AGR AUTH DATE","STATUS_DESC","BRANCHDESC","Region","State","Ticket size_bin", "Zone", "Area", "BCG INDUSTRYDESC","BCG SUB INDUSTRYDESC","PROMOTIONDESC"]]
# print(df2.shape)
# df2.head(3)
# df= df2.merge(df1, how='inner', on='AGREEMENTNO')
# df.shape

In [4]:
# Filtering active and closed cases
df= df.loc[df['STATUS_DESC'].str.contains('ACTIVE|CLOSED')]
print(df.shape)
df.head(3)

(183654, 15)


,AGREEMENTNO,EFF_RATE,LTV_bin,AMTFIN,AGR AUTH DATE,STATUS_DESC,BRANCHDESC,Region,State,Ticket size_bin,Zone,Area,BCG INDUSTRYDESC,BCG SUB INDUSTRYDESC,PROMOTIONDESC
0,X0HECHE00000016515,12.00,4. 50%-60%,2400000,26/10/2006,CLOSED,CHENNAI ONE HE,TAMIL NADU,TAMIL NADU,2. 20L-50L,SOUTH 1,CHENNAI 1 AREA,Consultancy / Professional Services,Photo studio and AV,NaN
1,X0HECHE00000016528,14.85,4. 50%-60%,2400000,26/10/2006,CLOSED,CHENNAI ONE HE,TAMIL NADU,TAMIL NADU,2. 20L-50L,SOUTH 1,CHENNAI 1 AREA,Consultancy / Professional Services,Photo studio and AV,NaN
2,X0HECHE00000021140,15.15,2. 20%-40%,600000,14/11/2006,CLOSED,CHENNAI ONE HE,TAMIL NADU,TAMIL NADU,1. <=20L,SOUTH 1,CHENNAI 1 AREA,Consultancy / Professional Services,Photo studio and AV,NaN


In [5]:
df['AGR AUTH DATE'] = pd.to_datetime(df['AGR AUTH DATE'])

# Filter rows where 'AGR_AUTH_DATE' is greater than 1st April 2021
df = df[df['AGR AUTH DATE'] >= pd.to_datetime('2023-04-01', format='%Y-%m-%d')]
df["AGR AUTH DATE"].min()

Timestamp('2023-04-11 00:00:00')

In [6]:
df["AGR AUTH DATE"].max()

Timestamp('2025-11-30 00:00:00')

In [7]:
df.shape

(80405, 15)

### Loading All_cols Deliq file

In [8]:
dl= pd.read_csv(r"D:\TASK\HE_HL_allcols_dump_Rcode\output HE & HL\HE_HL_deldump_cash_flow_mar21_nov25_AllCol.csv", encoding='latin1') #CUSTOMERNAME
print(dl.shape)
dl.head(3)

(8003158, 135)


,EMI_STARTDATE,LAST_EMI_DATE,EXPIRYDT,INTCOMP_BILLED,INTCOMP_RECD,ACCRUEDAMT,SECURITSATION_BANK,HYPOTHECATION_BANK,AGREEMENTNO,PROPOSALID,...,ADMIN.AND.PROCESSING.FEE.DUE,ADMIN.AND.PROCESSING.FEE.DPD,LEGAL.OR.RECOVERY.CHARGES,SD_AMT,SWITCH_CHARGES,EOM_MONTH,src,bkt,DPD,OD_PLUS_POS
0,05/09/2025,05/08/2026,05/09/2026,6047,6047,"17,489",NaN,LNB/2023-24/1049,EF01ABA0000067662,"8,00,26,311",...,NaN,NaN,NaN,NaN,NaN,Aug-25,ENCORE,0.0,0,1212735.0
1,05/09/2025,05/08/2026,05/09/2026,0,0,"4,319",NaN,NaN,EF01ABA0000067662,"8,00,26,311",...,NaN,NaN,NaN,NaN,NaN,Jul-25,ENCORE,0.0,0,1212735.0
2,05-09-2025,05-08-2026,05-09-2026,42354,42354,51030,NaN,LNB/2023-24/1049,EF01ABA0000067662,80026311,...,NaN,NaN,NaN,NaN,NaN,NOV-25,ENCORE,0.0,0,924088.0


### EMI_STARTDATE Mapping¶

In [9]:
# Taking recent month emi date and mapping against agreementno's

dl['EMI_STARTDATE']= pd.to_datetime(dl['EMI_STARTDATE'],  errors='coerce')
dl['EMI_STARTDATE']= dl['EMI_STARTDATE'].dt.strftime('%Y-%m-%d')

emi1= dl.loc[(dl['month']=='nov-2025'), ['AGREEMENTNO','EMI_STARTDATE']]
rough1= emi1.merge(df['AGREEMENTNO'], on='AGREEMENTNO', how='right')
print(rough1.shape)
rough1.head(3)

(80405, 2)


,AGREEMENTNO,EMI_STARTDATE
0,HE01AAJ00000041667,NaN
1,HE01AAJ00000041684,NaN
2,HE01AAJ00000041964,NaN


In [10]:
# mapping the emi_start date for closed cases agreements


emi2 = dl.drop_duplicates(subset='AGREEMENTNO', keep='last')
emi2= emi2[['AGREEMENTNO','EMI_STARTDATE']]
emi2.shape

(395581, 2)

In [11]:
rough2= rough1.merge(emi2, on='AGREEMENTNO', how='left')
print(rough2.shape)
rough2.head(3)

(80405, 3)


,AGREEMENTNO,EMI_STARTDATE_x,EMI_STARTDATE_y
0,HE01AAJ00000041667,NaN,2023-05-06
1,HE01AAJ00000041684,NaN,2023-05-08
2,HE01AAJ00000041964,NaN,2023-05-06


In [12]:
rough2.isna().sum()

AGREEMENTNO            0
EMI_STARTDATE_x    80405
EMI_STARTDATE_y     6429
dtype: int64

In [13]:
rough2['EMI_STARTDATE_x'] = rough2.apply(lambda row: row['EMI_STARTDATE_y'] if pd.isnull(row['EMI_STARTDATE_x']) else row['EMI_STARTDATE_x'], axis=1)
rough2.isna().sum()

AGREEMENTNO           0
EMI_STARTDATE_x    6429
EMI_STARTDATE_y    6429
dtype: int64

In [14]:
emi= rough2[['AGREEMENTNO','EMI_STARTDATE_x']]
col={'EMI_STARTDATE_x':'EMI_STARTDATE'}
emi= emi.rename(columns=col)
emi.head(3)

,AGREEMENTNO,EMI_STARTDATE
0,HE01AAJ00000041667,2023-05-06
1,HE01AAJ00000041684,2023-05-08
2,HE01AAJ00000041964,2023-05-06


In [15]:
#"PROP_TYPE","PROP_DESC"

data= df.merge(emi, how='left', on='AGREEMENTNO')
data.shape

(80405, 16)

### Taking PRODUCT SCHEME, DISB_STATUS from Deliq all cols

In [16]:
# mapping the product, scheme column against agreements

dfs1 = dl.drop_duplicates(subset='AGREEMENTNO', keep='last')
data = data.merge(dfs1[['AGREEMENTNO','PRODUCT', 'SCHEME','CUSTOMERNAME']], on='AGREEMENTNO', how='left')

In [17]:
# mapping the disb_status column

dfs2 = dl.loc[(dl['month']=='nov-2025'), ['AGREEMENTNO','DISB_STATUS']]
data= data.merge(dfs2[['AGREEMENTNO','DISB_STATUS']], on='AGREEMENTNO', how='left')

In [18]:
data = data.merge(dfs1[['AGREEMENTNO', 'DISB_STATUS']],  on='AGREEMENTNO', how='left')
data['DISB_STATUS_x'] = data.apply(lambda row: row['DISB_STATUS_y'] if pd.isnull(row['DISB_STATUS_x']) else row['DISB_STATUS_x'], axis=1)
data= data.drop('DISB_STATUS_y', axis=1)
col={'DISB_STATUS_x':'DISBS_STATUS'}
data= data.rename(columns=col)
data.head(3)

,AGREEMENTNO,EFF_RATE,LTV_bin,AMTFIN,AGR AUTH DATE,STATUS_DESC,BRANCHDESC,Region,State,Ticket size_bin,Zone,Area,BCG INDUSTRYDESC,BCG SUB INDUSTRYDESC,PROMOTIONDESC,EMI_STARTDATE,PRODUCT,SCHEME,CUSTOMERNAME,DISBS_STATUS
0,HE01AAJ00000041667,14.0,3. 40%-50%,100000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,EAST,MUZAFFARPUR AREA,Retail / Wholesale Trade,"Agriculture, forestry, fishing and related act...",Registered Mortgage,2023-05-06,LAP,Loan Against Property,ASHOK TIWARI NAGINA TIWARI,FULLY DISBURSED
1,HE01AAJ00000041684,13.0,3. 40%-50%,2800000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,2. 20L-50L,EAST,MUZAFFARPUR AREA,Retail / Wholesale Trade,Readymade Garments/Clothes,BT,2023-05-08,LAP,Loan Against Property,MD ALAM MD JAN,FULLY DISBURSED
2,HE01AAJ00000041964,12.0,3. 40%-50%,2000000,2023-04-30,CLOSED,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,EAST,MUZAFFARPUR AREA,Retail / Wholesale Trade,General Merchandise Store,Fresh Loans,2023-05-06,LAP,Loan Against Property,SUSHIL KUMAR GAURI SAH,FULLY DISBURSED


In [19]:
data.isnull().sum()

AGREEMENTNO                 0
EFF_RATE                    0
LTV_bin                     0
AMTFIN                      0
AGR AUTH DATE               0
STATUS_DESC                 0
BRANCHDESC                  0
Region                      0
State                       0
Ticket size_bin             0
Zone                        0
Area                        0
BCG INDUSTRYDESC         9737
BCG SUB INDUSTRYDESC    14413
PROMOTIONDESC               1
EMI_STARTDATE            6429
PRODUCT                     0
SCHEME                      0
CUSTOMERNAME                0
DISBS_STATUS                0
dtype: int64

In [20]:
column={'EFF_RATE':'CUSTOMER_IRR', 'STATUS_DESC':'STATUS',  'AGR AUTH DATE':'AGR_AUTH_DATE'} 

data= data.rename(columns=column)
data.head(3)

,AGREEMENTNO,CUSTOMER_IRR,LTV_bin,AMTFIN,AGR_AUTH_DATE,STATUS,BRANCHDESC,Region,State,Ticket size_bin,Zone,Area,BCG INDUSTRYDESC,BCG SUB INDUSTRYDESC,PROMOTIONDESC,EMI_STARTDATE,PRODUCT,SCHEME,CUSTOMERNAME,DISBS_STATUS
0,HE01AAJ00000041667,14.0,3. 40%-50%,100000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,EAST,MUZAFFARPUR AREA,Retail / Wholesale Trade,"Agriculture, forestry, fishing and related act...",Registered Mortgage,2023-05-06,LAP,Loan Against Property,ASHOK TIWARI NAGINA TIWARI,FULLY DISBURSED
1,HE01AAJ00000041684,13.0,3. 40%-50%,2800000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,2. 20L-50L,EAST,MUZAFFARPUR AREA,Retail / Wholesale Trade,Readymade Garments/Clothes,BT,2023-05-08,LAP,Loan Against Property,MD ALAM MD JAN,FULLY DISBURSED
2,HE01AAJ00000041964,12.0,3. 40%-50%,2000000,2023-04-30,CLOSED,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,EAST,MUZAFFARPUR AREA,Retail / Wholesale Trade,General Merchandise Store,Fresh Loans,2023-05-06,LAP,Loan Against Property,SUSHIL KUMAR GAURI SAH,FULLY DISBURSED


In [21]:
data['DISBS_STATUS'].value_counts()

DISBS_STATUS
FULLY DISBURSED        78285
PARTIALLY DISBURSED     2120
Name: count, dtype: int64

### Work on the EMI_STARTDATE for partially disbursed cases & null

In [22]:
data['AGR_AUTH_DATE']= pd.to_datetime(data['AGR_AUTH_DATE'], format="%Y-%m-%d")
data['EMI_STARTDATE']= pd.to_datetime(data['EMI_STARTDATE'], format="%Y-%d-%m").dt.strftime('%Y-%m-%d')

In [23]:
## Code to create a one month ahead from agr_auth_date

data['AGR_AUTH_DATE']= pd.to_datetime(data['AGR_AUTH_DATE'], format="%Y-%m-%d")

def calculate_one_month_ahead(date):
    next_month = date + timedelta(days=30)
    next_month_5th = datetime(next_month.year, next_month.month, 5)
    return next_month_5th

data['one_month_ahead_flag'] = data['AGR_AUTH_DATE'].apply(calculate_one_month_ahead)

#data['one_month_ahead_flag'] = data['AGR_AUTH_DATE']+ pd.DateOffset(months=1)

data.head(3)

,AGREEMENTNO,CUSTOMER_IRR,LTV_bin,AMTFIN,AGR_AUTH_DATE,STATUS,BRANCHDESC,Region,State,Ticket size_bin,...,Area,BCG INDUSTRYDESC,BCG SUB INDUSTRYDESC,PROMOTIONDESC,EMI_STARTDATE,PRODUCT,SCHEME,CUSTOMERNAME,DISBS_STATUS,one_month_ahead_flag
0,HE01AAJ00000041667,14.0,3. 40%-50%,100000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,...,MUZAFFARPUR AREA,Retail / Wholesale Trade,"Agriculture, forestry, fishing and related act...",Registered Mortgage,2023-06-05,LAP,Loan Against Property,ASHOK TIWARI NAGINA TIWARI,FULLY DISBURSED,2023-05-05
1,HE01AAJ00000041684,13.0,3. 40%-50%,2800000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,2. 20L-50L,...,MUZAFFARPUR AREA,Retail / Wholesale Trade,Readymade Garments/Clothes,BT,2023-08-05,LAP,Loan Against Property,MD ALAM MD JAN,FULLY DISBURSED,2023-05-05
2,HE01AAJ00000041964,12.0,3. 40%-50%,2000000,2023-04-30,CLOSED,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,...,MUZAFFARPUR AREA,Retail / Wholesale Trade,General Merchandise Store,Fresh Loans,2023-06-05,LAP,Loan Against Property,SUSHIL KUMAR GAURI SAH,FULLY DISBURSED,2023-05-05


In [24]:
# For EMI StartDate null cases and DISB_STATUS 'partially Disbursed' mark one_month_ahead_flag as EMI STARTDATE 

data['EMI_STARTDATE']= pd.to_datetime(data['EMI_STARTDATE'], format="%Y-%m-%d")
data['one_month_ahead_flag']= pd.to_datetime(data['one_month_ahead_flag'], format="%Y-%m-%d")

data['EMI_STARTDATE']= data['EMI_STARTDATE'].fillna(data['one_month_ahead_flag'])


In [25]:
data.isna().sum()

AGREEMENTNO                 0
CUSTOMER_IRR                0
LTV_bin                     0
AMTFIN                      0
AGR_AUTH_DATE               0
STATUS                      0
BRANCHDESC                  0
Region                      0
State                       0
Ticket size_bin             0
Zone                        0
Area                        0
BCG INDUSTRYDESC         9737
BCG SUB INDUSTRYDESC    14413
PROMOTIONDESC               1
EMI_STARTDATE               0
PRODUCT                     0
SCHEME                      0
CUSTOMERNAME                0
DISBS_STATUS                0
one_month_ahead_flag        0
dtype: int64

In [26]:
data['DISBS_STATUS']= data['DISBS_STATUS'].fillna('FULLY DISBURSED')
data['PRODUCT']= data['PRODUCT'].fillna('LAP')
data['SCHEME']= data['SCHEME'].fillna('HOME Equity-Finone')

In [27]:
data= data.fillna('Unmapped')

In [28]:
data= data.drop(['one_month_ahead_flag'], axis=1)
data.shape

(80405, 20)

In [29]:
# data['PRODUCT']= data['PRODUCT'].replace('Unmapped','LAP')
# data['SCHEME']= data['SCHEME'].replace('LAP','HOME Equity-Finone')

In [30]:
data['PRODUCT'].value_counts()

PRODUCT
LAP     53233
MLAP    27172
Name: count, dtype: int64

In [31]:
data.to_csv(r'D:\TASK\HL & LAP\LAP\LAP Staticpool\LAP\final.csv', index=False)
print("saved successfully!")

saved successfully!


In [32]:
#Loading final
final= pd.read_csv(r'D:\TASK\HL & LAP\LAP\LAP Staticpool\LAP\final.csv')

### Base month workings

In [33]:
# load the master data
result = pyreadr.read_r("D:\TASK\HL & LAP\HL\HL files\HE_HL_Deliq\HE_HL_nov25_DELDUMP_F.RDATA") 
master = result['finheconsolidated']

In [34]:
base_month = master[['AGREEMENTNO', 'DPD', 'OD_PLUS_POS', 'month']]
base_month.head(5)

,AGREEMENTNO,DPD,OD_PLUS_POS,month
rownames,,,,
1,XSEGCHE00000000110,25.0,0.0,jan-2007
2,XSEGBAN00000000303,25.0,0.0,jan-2007
3,XSEGBAN00000000769,0.0,0.0,jan-2007
4,XSEGBAN00000000888,25.0,0.0,jan-2007
5,XSEGCHE00000002732,0.0,0.0,jan-2007


In [35]:
base_month.isnull().sum()

AGREEMENTNO         0
DPD            151921
OD_PLUS_POS        10
month               0
dtype: int64

In [36]:
base = base_month.merge(final[["AGREEMENTNO","EMI_STARTDATE"]], on= "AGREEMENTNO", how = "inner")
base.shape

(1129836, 5)

In [37]:
base.head(3)

,AGREEMENTNO,DPD,OD_PLUS_POS,month,EMI_STARTDATE
0,HE01UDI00000041452,0.0,10000000.0,apr-2023,2023-06-05
1,HE01COI00000041453,0.0,17000000.0,apr-2023,2023-06-05
2,HE01YAM00000041540,0.0,1450000.0,apr-2023,2023-06-05


In [38]:
base_month = base.copy()
base_month['month_date']= pd.to_datetime(base_month['month'], format="%b-%Y")
base_month['month_date'] = base_month['month_date'].dt.strftime('%d/%m/%Y')
base_month.head()

,AGREEMENTNO,DPD,OD_PLUS_POS,month,EMI_STARTDATE,month_date
0,HE01UDI00000041452,0.0,10000000.0,apr-2023,2023-06-05,01/04/2023
1,HE01COI00000041453,0.0,17000000.0,apr-2023,2023-06-05,01/04/2023
2,HE01YAM00000041540,0.0,1450000.0,apr-2023,2023-06-05,01/04/2023
3,HE01ALO00000041541,0.0,17500000.0,apr-2023,2023-06-05,01/04/2023
4,HE01JRH00000041542,0.0,5500000.0,apr-2023,2023-06-05,01/04/2023


In [39]:
base_month['month_date'] = pd.to_datetime(base_month['month_date'], format="%d/%m/%Y")
base_month['EMI_STARTDATE'] = pd.to_datetime(base_month['EMI_STARTDATE'], format="%Y-%m-%d")   
base_month['mob'] = ((base_month['month_date'].dt.month - base_month['EMI_STARTDATE'].dt.month)+
                     (base_month['month_date'].dt.year - base_month['EMI_STARTDATE'].dt.year)*12)

In [40]:
base_month.head()

,AGREEMENTNO,DPD,OD_PLUS_POS,month,EMI_STARTDATE,month_date,mob
0,HE01UDI00000041452,0.0,10000000.0,apr-2023,2023-06-05,2023-04-01,-2
1,HE01COI00000041453,0.0,17000000.0,apr-2023,2023-06-05,2023-04-01,-2
2,HE01YAM00000041540,0.0,1450000.0,apr-2023,2023-06-05,2023-04-01,-2
3,HE01ALO00000041541,0.0,17500000.0,apr-2023,2023-06-05,2023-04-01,-2
4,HE01JRH00000041542,0.0,5500000.0,apr-2023,2023-06-05,2023-04-01,-2


In [41]:
base_month = base_month.loc[base_month.mob >= 0 ]
base_month= base_month.fillna(0)
base_month.to_csv("D:\\TASK\\HL & LAP\\LAP\\LAP Staticpool\\LAP\\base_month.csv", index = False)

In [42]:
base_month.mob.unique()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29])

In [43]:
base_month.mob.max()

29

In [44]:
base_month.isnull().sum()

AGREEMENTNO      0
DPD              0
OD_PLUS_POS      0
month            0
EMI_STARTDATE    0
month_date       0
mob              0
dtype: int64

In [45]:
# Read CSV files
final = final.drop_duplicates()

base_month = base_month[["AGREEMENTNO","DPD","OD_PLUS_POS","mob"]]
final.head(3)

,AGREEMENTNO,CUSTOMER_IRR,LTV_bin,AMTFIN,AGR_AUTH_DATE,STATUS,BRANCHDESC,Region,State,Ticket size_bin,Zone,Area,BCG INDUSTRYDESC,BCG SUB INDUSTRYDESC,PROMOTIONDESC,EMI_STARTDATE,PRODUCT,SCHEME,CUSTOMERNAME,DISBS_STATUS
0,HE01AAJ00000041667,14.0,3. 40%-50%,100000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,EAST,MUZAFFARPUR AREA,Retail / Wholesale Trade,"Agriculture, forestry, fishing and related act...",Registered Mortgage,2023-06-05,LAP,Loan Against Property,ASHOK TIWARI NAGINA TIWARI,FULLY DISBURSED
1,HE01AAJ00000041684,13.0,3. 40%-50%,2800000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,2. 20L-50L,EAST,MUZAFFARPUR AREA,Retail / Wholesale Trade,Readymade Garments/Clothes,BT,2023-08-05,LAP,Loan Against Property,MD ALAM MD JAN,FULLY DISBURSED
2,HE01AAJ00000041964,12.0,3. 40%-50%,2000000,2023-04-30,CLOSED,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,EAST,MUZAFFARPUR AREA,Retail / Wholesale Trade,General Merchandise Store,Fresh Loans,2023-06-05,LAP,Loan Against Property,SUSHIL KUMAR GAURI SAH,FULLY DISBURSED


In [46]:
final['AGR_AUTH_DATE'] = pd.to_datetime(final['AGR_AUTH_DATE'])

# Filter rows where 'AGR_AUTH_DATE' is greater than 1st April 2023
final = final[final['AGR_AUTH_DATE'] >= pd.to_datetime('2023-04-01', format='%Y-%m-%d')]
final.AGR_AUTH_DATE.min()

Timestamp('2023-04-11 00:00:00')

In [47]:
final.AGR_AUTH_DATE.max()

Timestamp('2025-11-30 00:00:00')

In [48]:
final.shape

(80405, 20)

In [49]:
final['AGR_AUTH_DATE']= pd.to_datetime(final['AGR_AUTH_DATE'], format='%Y-%m-%d')
final['EMI_STARTDATE']= pd.to_datetime(final['EMI_STARTDATE'], format='%Y-%m-%d')
final.head(3)

,AGREEMENTNO,CUSTOMER_IRR,LTV_bin,AMTFIN,AGR_AUTH_DATE,STATUS,BRANCHDESC,Region,State,Ticket size_bin,Zone,Area,BCG INDUSTRYDESC,BCG SUB INDUSTRYDESC,PROMOTIONDESC,EMI_STARTDATE,PRODUCT,SCHEME,CUSTOMERNAME,DISBS_STATUS
0,HE01AAJ00000041667,14.0,3. 40%-50%,100000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,EAST,MUZAFFARPUR AREA,Retail / Wholesale Trade,"Agriculture, forestry, fishing and related act...",Registered Mortgage,2023-06-05,LAP,Loan Against Property,ASHOK TIWARI NAGINA TIWARI,FULLY DISBURSED
1,HE01AAJ00000041684,13.0,3. 40%-50%,2800000,2023-04-30,ACTIVE,MOTIHARI HE,BIHAR,BIHAR,2. 20L-50L,EAST,MUZAFFARPUR AREA,Retail / Wholesale Trade,Readymade Garments/Clothes,BT,2023-08-05,LAP,Loan Against Property,MD ALAM MD JAN,FULLY DISBURSED
2,HE01AAJ00000041964,12.0,3. 40%-50%,2000000,2023-04-30,CLOSED,MOTIHARI HE,BIHAR,BIHAR,1. <=20L,EAST,MUZAFFARPUR AREA,Retail / Wholesale Trade,General Merchandise Store,Fresh Loans,2023-06-05,LAP,Loan Against Property,SUSHIL KUMAR GAURI SAH,FULLY DISBURSED


In [50]:
base_month.head(3)

,AGREEMENTNO,DPD,OD_PLUS_POS,mob
2053,HE01BAN00000041922,0.0,8000000.0,0
2143,HE01RTH00000042029,0.0,4100000.0,0
2150,HE01DRM00000042040,0.0,1600000.0,0


In [51]:
# Define DPD stages mapping function
def dpd_stage(dpd):
    if dpd > 90:
        return "stage 3"
    elif 30 < dpd <= 90:
        return "stage 2"
    else:
        return "stage 1"

# Process base_month data and merge with final

max= base_month.mob.max()

for m in range(max+1):
    mob = base_month[base_month['mob'] == m].copy()
    mob['DPD'] = mob['DPD'].astype(int)
    mob[str(m) + 'M_curr'] = mob['DPD'].apply(dpd_stage)

    final = final.merge(mob[['AGREEMENTNO', str(m) + 'M_curr']], on='AGREEMENTNO', how='left')

    print(m)


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29


In [52]:
# Save the final dataframe to CSV if needed
final.drop_duplicates().to_csv(r"D:\TASK\HL & LAP\LAP\LAP Staticpool\LAP\LAP_SPA.csv", index=False)

In [53]:
final.columns

Index(['AGREEMENTNO', 'CUSTOMER_IRR', 'LTV_bin', 'AMTFIN', 'AGR_AUTH_DATE',
       'STATUS', 'BRANCHDESC', 'Region', 'State', 'Ticket size_bin', 'Zone',
       'Area', 'BCG INDUSTRYDESC', 'BCG SUB INDUSTRYDESC', 'PROMOTIONDESC',
       'EMI_STARTDATE', 'PRODUCT', 'SCHEME', 'CUSTOMERNAME', 'DISBS_STATUS',
       '0M_curr', '1M_curr', '2M_curr', '3M_curr', '4M_curr', '5M_curr',
       '6M_curr', '7M_curr', '8M_curr', '9M_curr', '10M_curr', '11M_curr',
       '12M_curr', '13M_curr', '14M_curr', '15M_curr', '16M_curr', '17M_curr',
       '18M_curr', '19M_curr', '20M_curr', '21M_curr', '22M_curr', '23M_curr',
       '24M_curr', '25M_curr', '26M_curr', '27M_curr', '28M_curr', '29M_curr'],
      dtype='object')

In [54]:
final.isnull().sum()

AGREEMENTNO                 0
CUSTOMER_IRR                0
LTV_bin                     0
AMTFIN                      0
AGR_AUTH_DATE               0
STATUS                      0
BRANCHDESC                  0
Region                      0
State                       0
Ticket size_bin             0
Zone                        0
Area                        0
BCG INDUSTRYDESC            0
BCG SUB INDUSTRYDESC        0
PROMOTIONDESC               0
EMI_STARTDATE               0
PRODUCT                     0
SCHEME                      0
CUSTOMERNAME                0
DISBS_STATUS                0
0M_curr                  5768
1M_curr                  9720
2M_curr                 13393
3M_curr                 16985
4M_curr                 20082
5M_curr                 23263
6M_curr                 25833
7M_curr                 29639
8M_curr                 32832
9M_curr                 36098
10M_curr                39242
11M_curr                42182
12M_curr                45148
13M_curr  

In [55]:
final.shape

(80405, 50)

In [56]:
final.PRODUCT.value_counts()

PRODUCT
LAP     53233
MLAP    27172
Name: count, dtype: int64

In [57]:
# Function to remove the "1." and "2." prefixes in Ticket Size Bin column

def remove_prefix(entry):
    return re.sub(r'\b[1-9]\.', '', entry)

# Apply the function to the specified column
final['Ticket size_bin'] = final['Ticket size_bin'].apply(lambda x: remove_prefix(x))

In [58]:
final['Ticket size_bin'].value_counts()

Ticket size_bin
<=20L       33504
20L-50L     25431
>100L       10053
50L-75L      6884
75L-100L     4533
Name: count, dtype: int64

In [59]:
final['LTV_bin'] = final['LTV_bin'].apply(lambda x: remove_prefix(str(x)))

In [60]:
final['LTV_bin']= final['LTV_bin'].fillna('Unmapped')

In [61]:
final['LTV_bin'].value_counts()

LTV_bin
50%-60%    21555
40%-50%    20834
20%-40%    20504
60%-70%    14020
<=20%       3486
>70%           6
Name: count, dtype: int64

In [62]:
final.columns

Index(['AGREEMENTNO', 'CUSTOMER_IRR', 'LTV_bin', 'AMTFIN', 'AGR_AUTH_DATE',
       'STATUS', 'BRANCHDESC', 'Region', 'State', 'Ticket size_bin', 'Zone',
       'Area', 'BCG INDUSTRYDESC', 'BCG SUB INDUSTRYDESC', 'PROMOTIONDESC',
       'EMI_STARTDATE', 'PRODUCT', 'SCHEME', 'CUSTOMERNAME', 'DISBS_STATUS',
       '0M_curr', '1M_curr', '2M_curr', '3M_curr', '4M_curr', '5M_curr',
       '6M_curr', '7M_curr', '8M_curr', '9M_curr', '10M_curr', '11M_curr',
       '12M_curr', '13M_curr', '14M_curr', '15M_curr', '16M_curr', '17M_curr',
       '18M_curr', '19M_curr', '20M_curr', '21M_curr', '22M_curr', '23M_curr',
       '24M_curr', '25M_curr', '26M_curr', '27M_curr', '28M_curr', '29M_curr'],
      dtype='object')

In [63]:
final.to_csv(r"D:\TASK\HL & LAP\LAP\LAP Staticpool\LAP\LAP_SPA.csv", index=False)
print("Saved successfully!!")

Saved successfully!!
